In [ ]:
# import libraries
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) # display all columns in the dataframe

In [ ]:
patients = pd.read_csv('../data/processed/patients.csv')
vitals = pd.read_csv('../data/processed/vitals.csv')
history = pd.read_csv('../data/processed/history.csv')
labs = pd.read_csv('../data/processed/labs.csv')
outcomes = pd.read_csv('../data/processed/outcomes.csv')

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name} shape: {df.shape}")

In [ ]:
vitals.columns

In [ ]:
labs.columns

In [ ]:
vital_cols = [
  'heart_rate',
  'temperature',
  'oxygen_saturation',
  'respiratory_rate',
  'blood_pressure'
]

lab_cols = [
  'white_cell_count',
  'crp',
  'lactate',
  'creatinine',
  'platelet_count',
]

In [ ]:
outcomes.columns

In [ ]:
rng = np.random.default_rng(7) # set random seed for reproducibility

def get_prediction_time(row, vitals_df):
  if row['sepsis_event']:
    return row['diagnosis_time'] - pd.Timedelta(hours=9)
  pv = vitals_df[vitals_df['patient_id'] == row['patient_id']]
  start, end = pv['timestamp'].min(), pv['timestamp'].max()
  span_hours = max((end - start).total_seconds() / 3600, 1)
  offset_hours = rng.uniform(0.4, 0.9) * span_hours
  return start + pd.Timedelta(hours=offset_hours)

outcomes = outcomes.copy()
outcomes['prediction_time'] = outcomes.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head(15)

In [ ]:
LOOKBACK_HOURS = 6

def vitals_features(pid, cutoff, df):
  window=df[
    (df['patient_id'] == pid) & (df['timestamp'] <= cutoff) & (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
  ]

  if window.empty:
    window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
  feats={}
  for col in vital_cols:
    vals = window[col]
    feats[f'{col}_mean'] = vals.mean()
    feats[f'{col}_min'] = vals.min()
    feats[f'{col}_max'] = vals.max()
    feats[f'{col}_std'] = vals.std() if len(vals) > 1 else 0.0
    feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan

    if len(window) > 1:
      hours = (window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]).total_seconds() / 3600
      feats[f'{col}_rate_per_hour'] = (vals.iloc[-1] - vals.iloc[0]) / hours if hours > 0 else 0.0
    else:
      feats[f'{col}_rate_per_hour'] = 0.0
      
  return feats


In [ ]:
vitals_features_rows = [
  {'patient_id': pid, **vitals_features(pid, cutoff, vitals)}
  for pid, cutoff in zip(outcomes['patient_id'], outcomes['cutoff'])
]
vitals_features_df = pd.DataFrame(vitals_features_rows)
vitals_features_df.head(10)

In [ ]:
LAB_LOOKBACK_HOURS = 24

def labs_features(pid, cutoff, df):
  window=df[
    (df['patient_id'] == pid) & 
    (df['timestamp'] <= cutoff) & 
    (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
  ]

  if window.empty:
    window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
  feats={}
  for col in lab_cols:
    vals = window[col]
    feats[f'{col}_mean'] = vals.mean()
    feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan
  
  return feats

In [ ]:
labs_features_rows = [
  {'patient_id': pid, **labs_features(pid, cutoff, labs)}
  for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
labs_features_df = pd.DataFrame(labs_features_rows)
labs_features_df.head(10)

In [ ]:
patients.columns

In [ ]:
static = patients[['patient_id', 'age', 'gender']].copy()

static['comorbidity_count'] = patients[['medical_conditions']].apply(
  lambda x: 0 if pd.isna(x) or x == 'None reported'
  else len(x.split(','))
)

static = pd.get_dummies(static, columns=['gender'], drop_first=True)
static.head(10)

In [ ]:
feature = (
  static
  .merge(vitals_features_df, on='patient_id')
  .merge(labs_features_df, on='patient_id')
  .merge(outcomes[['patient_id', 'sepsis_event']], on='patient_id')
)

feature_columns = [
  c for c in feature.columns
  if c not in ('patient_id', 'sepsis_event')
]

numeric_cols = feature[feature_columns].select_dtypes(include='number').columns

feature[numeric_cols] = feature[numeric_cols].fillna(feature[numeric_cols].median())

feature['sepsis_event'] = feature['sepsis_event'].astype(int)

print(feature.shape)

In [ ]:
feature.head(10)

In [ ]:
feature.columns

In [ ]:
check_cols = ['heart_rate_last', 'oxygen_saturation_last', 'crp_last', 'lactate_last']
feature.groupby('sepsis_event')[check_cols].mean()

In [ ]:
corr = feature[feature_columns + ['sepsis_event']].corr()['sepsis_event'].drop('sepsis_event')
corr.sort_values(key=abs, ascending=False).head(10)

In [ ]:
feature.to_csv('../data/processed/sepsis_features.csv', index=False)